# Old Planner + Caller: split_200 pipeline validation
Inference only on the supplied split_200.txt (200 episodes; 168 overlap adapter training IDs). Unguided generation for both nodes, followed by JSON/schema validation. Maximum7 decisions and7 executed tool calls per episode. Two independently trained local adapters (Planner v4, Caller v5), exact English training prompts and TypeScript tool schemas. Two independent GPU workers, batch4 per GPU (up to8 workflows total). CUDA Graph enabled; prefix caching disabled on T4/SM75 because its Triton prefix-prefill kernel fails. Prefix caching remains enabled on SM80+. At most one revision per node per decision. First CUDA graph capture can add startup time. No silent eager fallback. Supplied ASR for current/history audio; no transcript oracle. No Summarizer generation; its route ends the workflow. Report local approximate contest score, tool-call F1, state match and per-sample traces. No public-test run or submission. Checkpoint file hashes and prompt/model/data provenance are verified before loading. Save batch outputs for continuation; no per-sample timeout.


In [ ]:
from pathlib import Path
import os,sys,json,hashlib,subprocess,zipfile,time,shutil
MODEL='Qwen/Qwen3-4B-Instruct-2507'
REVISION='cdbee75f17c01a7cc42f958dc650907174af0554'
MODE='multi_local'  # trained local Planner and Caller, terminal route without Summarizer
assert MODE=='multi_local'
RUN_NAME='old_adapters_unguided_limits7_split200'
SESSION_STARTED=time.monotonic()
SESSION_HOURS=10  # Stop before new batches after this budget; no per-sample timeout.
RESUME_DIR=None  # Optional mounted prior run directory containing config.json and split subfolders.
WORK=Path('/kaggle/working/pipeline_validation_200')/RUN_NAME;WORK.mkdir(parents=True,exist_ok=True)
if RESUME_DIR is not None:shutil.copytree(Path(RESUME_DIR),WORK,dirs_exist_ok=True)
SOURCE=Path('/kaggle/working/pipeline_validation_200_source')
CORE_REV='4894d331692293565635454a8c686b7c18314a04'
if not SOURCE.exists():subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/tnisl/baseline.git',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','origin',CORE_REV],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)


In [ ]:
PAYLOAD={'adapter_train_ids.txt': 'train_0002\ntrain_0003\ntrain_0005\ntrain_0006\ntrain_0007\ntrain_0008\ntrain_0009\ntrain_0011\ntrain_0012\ntrain_0013\ntrain_0014\ntrain_0018\ntrain_0019\ntrain_0020\ntrain_0025\ntrain_0026\ntrain_0027\ntrain_0028\ntrain_0029\ntrain_0030\ntrain_0032\ntrain_0033\ntrain_0034\ntrain_0035\ntrain_0036\ntrain_0037\ntrain_0038\ntrain_0039\ntrain_0040\ntrain_0041\ntrain_0042\ntrain_0043\ntrain_0045\ntrain_0046\ntrain_0047\ntrain_0048\ntrain_0049\ntrain_0050\ntrain_0052\ntrain_0055\ntrain_0057\ntrain_0059\ntrain_0060\ntrain_0061\ntrain_0062\ntrain_0064\ntrain_0065\ntrain_0068\ntrain_0069\ntrain_0070\ntrain_0071\ntrain_0072\ntrain_0074\ntrain_0076\ntrain_0078\ntrain_0079\ntrain_0081\ntrain_0082\ntrain_0083\ntrain_0084\ntrain_0085\ntrain_0086\ntrain_0087\ntrain_0089\ntrain_0091\ntrain_0092\ntrain_0093\ntrain_0094\ntrain_0096\ntrain_0097\ntrain_0098\ntrain_0100\ntrain_0103\ntrain_0104\ntrain_0105\ntrain_0107\ntrain_0108\ntrain_0109\ntrain_0111\ntrain_0112\ntrain_0113\ntrain_0115\ntrain_0116\ntrain_0119\ntrain_0121\ntrain_0122\ntrain_0123\ntrain_0124\ntrain_0125\ntrain_0126\ntrain_0127\ntrain_0128\ntrain_0129\ntrain_0131\ntrain_0132\ntrain_0134\ntrain_0135\ntrain_0136\ntrain_0137\ntrain_0138\ntrain_0139\ntrain_0140\ntrain_0142\ntrain_0143\ntrain_0146\ntrain_0148\ntrain_0149\ntrain_0150\ntrain_0151\ntrain_0152\ntrain_0153\ntrain_0154\ntrain_0155\ntrain_0156\ntrain_0157\ntrain_0158\ntrain_0159\ntrain_0161\ntrain_0165\ntrain_0166\ntrain_0168\ntrain_0170\ntrain_0171\ntrain_0173\ntrain_0174\ntrain_0176\ntrain_0177\ntrain_0178\ntrain_0179\ntrain_0180\ntrain_0182\ntrain_0183\ntrain_0184\ntrain_0185\ntrain_0186\ntrain_0187\ntrain_0188\ntrain_0189\ntrain_0190\ntrain_0191\ntrain_0192\ntrain_0193\ntrain_0195\ntrain_0197\ntrain_0199\ntrain_0200\ntrain_0201\ntrain_0202\ntrain_0204\ntrain_0205\ntrain_0206\ntrain_0208\ntrain_0209\ntrain_0210\ntrain_0211\ntrain_0212\ntrain_0213\ntrain_0215\ntrain_0217\ntrain_0220\ntrain_0221\ntrain_0222\ntrain_0224\ntrain_0226\ntrain_0227\ntrain_0229\ntrain_0230\ntrain_0232\ntrain_0234\ntrain_0235\ntrain_0237\ntrain_0238\ntrain_0239\ntrain_0241\ntrain_0242\ntrain_0243\ntrain_0244\ntrain_0245\ntrain_0246\ntrain_0247\ntrain_0248\ntrain_0249\ntrain_0250\ntrain_0251\ntrain_0252\ntrain_0253\ntrain_0254\ntrain_0256\ntrain_0257\ntrain_0258\ntrain_0259\ntrain_0260\ntrain_0261\ntrain_0262\ntrain_0263\ntrain_0264\ntrain_0265\ntrain_0266\ntrain_0267\ntrain_0268\ntrain_0269\ntrain_0270\ntrain_0271\ntrain_0272\ntrain_0273\ntrain_0274\ntrain_0275\ntrain_0276\ntrain_0277\ntrain_0279\ntrain_0280\ntrain_0281\ntrain_0282\ntrain_0283\ntrain_0284\ntrain_0286\ntrain_0287\ntrain_0288\ntrain_0289\ntrain_0290\ntrain_0291\ntrain_0292\ntrain_0293\ntrain_0294\ntrain_0295\ntrain_0296\ntrain_0297\ntrain_0298\ntrain_0299\ntrain_0300\ntrain_0301\ntrain_0302\ntrain_0303\ntrain_0306\ntrain_0307\ntrain_0308\ntrain_0309\ntrain_0310\ntrain_0311\ntrain_0312\ntrain_0314\ntrain_0315\ntrain_0317\ntrain_0318\ntrain_0319\ntrain_0320\ntrain_0322\ntrain_0323\ntrain_0324\ntrain_0325\ntrain_0326\ntrain_0327\ntrain_0328\ntrain_0329\ntrain_0331\ntrain_0332\ntrain_0333\ntrain_0334\ntrain_0335\ntrain_0336\ntrain_0337\ntrain_0338\ntrain_0339\ntrain_0340\ntrain_0341\ntrain_0342\ntrain_0343\ntrain_0347\ntrain_0348\ntrain_0349\ntrain_0351\ntrain_0352\ntrain_0353\ntrain_0355\ntrain_0356\ntrain_0358\ntrain_0359\ntrain_0360\ntrain_0361\ntrain_0362\ntrain_0363\ntrain_0364\ntrain_0365\ntrain_0366\ntrain_0367\ntrain_0368\ntrain_0369\ntrain_0370\ntrain_0371\ntrain_0372\ntrain_0373\ntrain_0374\ntrain_0375\ntrain_0376\ntrain_0377\ntrain_0379\ntrain_0380\ntrain_0382\ntrain_0383\ntrain_0384\ntrain_0385\ntrain_0386\ntrain_0387\ntrain_0388\ntrain_0389\ntrain_0390\ntrain_0391\ntrain_0392\ntrain_0393\ntrain_0394\ntrain_0396\ntrain_0397\ntrain_0398\ntrain_0399\ntrain_0400\ntrain_0401\ntrain_0402\ntrain_0403\ntrain_0404\ntrain_0406\ntrain_0407\ntrain_0408\ntrain_0409\ntrain_0410\ntrain_0411\ntrain_0412\ntrain_0413\ntrain_0414\ntrain_0415\ntrain_0416\ntrain_0417\ntrain_0418\ntrain_0419\ntrain_0420\ntrain_0421\ntrain_0423\ntrain_0424\ntrain_0425\ntrain_0426\ntrain_0427\ntrain_0428\ntrain_0429\ntrain_0430\ntrain_0431\ntrain_0432\ntrain_0433\ntrain_0434\ntrain_0436\ntrain_0437\ntrain_0439\ntrain_0440\ntrain_0441\ntrain_0443\ntrain_0444\ntrain_0445\ntrain_0446\ntrain_0447\ntrain_0448\ntrain_0449\ntrain_0450\ntrain_0451\ntrain_0452\ntrain_0454\ntrain_0455\ntrain_0456\ntrain_0458\ntrain_0459\ntrain_0460\ntrain_0461\ntrain_0462\ntrain_0463\ntrain_0464\ntrain_0465\ntrain_0466\ntrain_0467\ntrain_0470\ntrain_0471\ntrain_0472\ntrain_0473\ntrain_0474\ntrain_0475\ntrain_0477\ntrain_0478\ntrain_0479\ntrain_0482\ntrain_0483\ntrain_0484\ntrain_0485\ntrain_0488\ntrain_0490\ntrain_0491\ntrain_0493\ntrain_0494\ntrain_0495\ntrain_0496\ntrain_0498\ntrain_0499\ntrain_0501\ntrain_0502\ntrain_0503\ntrain_0504\ntrain_0506\ntrain_0507\ntrain_0509\ntrain_0510\ntrain_0511\ntrain_0512\ntrain_0513\ntrain_0514\ntrain_0515\ntrain_0516\ntrain_0517\ntrain_0520\ntrain_0523\ntrain_0526\ntrain_0527\ntrain_0528\ntrain_0529\ntrain_0530\ntrain_0531\ntrain_0532\ntrain_0533\ntrain_0534\ntrain_0535\ntrain_0537\ntrain_0539\ntrain_0540\ntrain_0541\ntrain_0543\ntrain_0544\ntrain_0545\ntrain_0546\ntrain_0548\ntrain_0551\ntrain_0556\ntrain_0557\ntrain_0558\ntrain_0559\ntrain_0562\ntrain_0565\ntrain_0567\ntrain_0568\ntrain_0570\ntrain_0571\ntrain_0573\ntrain_0574\ntrain_0575\ntrain_0578\ntrain_0579\ntrain_0581\ntrain_0583\ntrain_0584\ntrain_0587\ntrain_0588\ntrain_0589\ntrain_0593\ntrain_0594\ntrain_0595\ntrain_0597\ntrain_0599\ntrain_0600\ntrain_0601\ntrain_0602\ntrain_0603\ntrain_0604\ntrain_0605\ntrain_0606\ntrain_0607\ntrain_0608\ntrain_0609\ntrain_0610\ntrain_0611\ntrain_0612\ntrain_0613\ntrain_0614\ntrain_0615\ntrain_0616\ntrain_0617\ntrain_0619\ntrain_0620\ntrain_0622\ntrain_0623\ntrain_0626\ntrain_0627\ntrain_0629\ntrain_0630\ntrain_0631\ntrain_0632\ntrain_0633\ntrain_0634\ntrain_0635\ntrain_0636\ntrain_0637\ntrain_0638\ntrain_0639\ntrain_0640\ntrain_0641\ntrain_0642\ntrain_0643\ntrain_0645\ntrain_0647\ntrain_0648\ntrain_0651\ntrain_0652\ntrain_0653\ntrain_0654\ntrain_0655\ntrain_0657\ntrain_0660\ntrain_0665\ntrain_0666\ntrain_0668\ntrain_0669\ntrain_0671\ntrain_0672\ntrain_0673\ntrain_0674\ntrain_0675\ntrain_0676\ntrain_0677\ntrain_0678\ntrain_0679\ntrain_0680\ntrain_0681\ntrain_0682\ntrain_0683\ntrain_0684\ntrain_0685\ntrain_0686\ntrain_0687\ntrain_0688\ntrain_0690\ntrain_0691\ntrain_0692\ntrain_0693\ntrain_0695\ntrain_0696\ntrain_0697\ntrain_0698\ntrain_0699\ntrain_0700\ntrain_0701\ntrain_0702\ntrain_0703\ntrain_0704\ntrain_0705\ntrain_0706\ntrain_0707\ntrain_0708\ntrain_0709\ntrain_0710\ntrain_0711\ntrain_0712\ntrain_0713\ntrain_0714\ntrain_0715\ntrain_0716\ntrain_0717\ntrain_0720\ntrain_0721\ntrain_0724\ntrain_0725\ntrain_0728\ntrain_0730\ntrain_0731\ntrain_0732\ntrain_0734\ntrain_0735\ntrain_0736\ntrain_0738\ntrain_0739\ntrain_0740\ntrain_0741\ntrain_0743\ntrain_0744\ntrain_0745\ntrain_0746\ntrain_0747\ntrain_0748\ntrain_0749\ntrain_0751\ntrain_0753\ntrain_0755\ntrain_0757\ntrain_0760\ntrain_0761\ntrain_0765\ntrain_0766\ntrain_0767\ntrain_0769\ntrain_0771\ntrain_0773\ntrain_0774\ntrain_0775\ntrain_0776\ntrain_0778\ntrain_0780\ntrain_0781\ntrain_0782\ntrain_0783\ntrain_0784\ntrain_0785\ntrain_0786\ntrain_0788\ntrain_0789\ntrain_0791\ntrain_0792\ntrain_0793\ntrain_0794\ntrain_0795\ntrain_0796\ntrain_0797\ntrain_0798\ntrain_0799\ntrain_0801\ntrain_0802\ntrain_0803\ntrain_0804\ntrain_0806\ntrain_0807\ntrain_0808\ntrain_0809\ntrain_0810\ntrain_0811\ntrain_0812\ntrain_0813\ntrain_0816\ntrain_0817\ntrain_0821\ntrain_0822\ntrain_0823\ntrain_0824\ntrain_0825\ntrain_0826\ntrain_0827\ntrain_0828\ntrain_0829\ntrain_0830\ntrain_0831\ntrain_0832\ntrain_0833\ntrain_0835\ntrain_0836\ntrain_0837\ntrain_0838\ntrain_0839\ntrain_0840\ntrain_0841\ntrain_0842\ntrain_0843\ntrain_0846\ntrain_0847\ntrain_0848\ntrain_0849\ntrain_0850\ntrain_0851\ntrain_0852\ntrain_0853\ntrain_0854\ntrain_0855\ntrain_0856\ntrain_0858\ntrain_0859\ntrain_0860\ntrain_0861\ntrain_0862\ntrain_0863\ntrain_0865\ntrain_0866\ntrain_0868\ntrain_0869\ntrain_0871\ntrain_0872\ntrain_0873\ntrain_0874\ntrain_0876\ntrain_0877\ntrain_0878\ntrain_0879\ntrain_0880\ntrain_0881\ntrain_0882\ntrain_0883\ntrain_0884\ntrain_0885\ntrain_0886\ntrain_0887\ntrain_0888\ntrain_0889\ntrain_0890\ntrain_0891\ntrain_0892\ntrain_0893\ntrain_0894\ntrain_0895\ntrain_0896\ntrain_0897\ntrain_0898\ntrain_0899\ntrain_0900\ntrain_0901\ntrain_0902\ntrain_0903\ntrain_0904\ntrain_0905\ntrain_0906\ntrain_0907\ntrain_0908\ntrain_0909\ntrain_0910\ntrain_0911\ntrain_0912\ntrain_0913\ntrain_0915\ntrain_0919\ntrain_0920\ntrain_0921\ntrain_0922\ntrain_0925\ntrain_0926\ntrain_0929\ntrain_0930\ntrain_0931\ntrain_0932\ntrain_0934\ntrain_0936\ntrain_0937\ntrain_0938\ntrain_0939\ntrain_0940\ntrain_0942\ntrain_0944\ntrain_0946\ntrain_0949\ntrain_0950\ntrain_0951\ntrain_0952\ntrain_0955\ntrain_0956\ntrain_0957\ntrain_0961\ntrain_0962\ntrain_0963\ntrain_0964\ntrain_0965\ntrain_0966\ntrain_0967\ntrain_0968\ntrain_0969\ntrain_0970\ntrain_0971\ntrain_0972\ntrain_0973\ntrain_0974\ntrain_0975\ntrain_0976\ntrain_0979\ntrain_0980\ntrain_0981\ntrain_0982\ntrain_0983\ntrain_0984\ntrain_0986\ntrain_0988\ntrain_0990\ntrain_0992\ntrain_0993\ntrain_0994\ntrain_0995\ntrain_0996\ntrain_0997\ntrain_0998\ntrain_0999\ntrain_1002\ntrain_1003\ntrain_1004\ntrain_1005\ntrain_1006\ntrain_1009\ntrain_1010\ntrain_1011\ntrain_1012\ntrain_1013\ntrain_1015\ntrain_1017\ntrain_1020\ntrain_1022\ntrain_1023\ntrain_1024\ntrain_1025\ntrain_1026\ntrain_1027\ntrain_1028\ntrain_1029\ntrain_1030\ntrain_1031\ntrain_1032\ntrain_1033\ntrain_1034\ntrain_1035\ntrain_1036\ntrain_1038\ntrain_1039\ntrain_1040\ntrain_1041\ntrain_1044\ntrain_1046\ntrain_1050\ntrain_1052\ntrain_1053\ntrain_1054\ntrain_1055\ntrain_1056\ntrain_1057\ntrain_1058\ntrain_1059\ntrain_1061\ntrain_1063\ntrain_1064\ntrain_1066\ntrain_1067\ntrain_1068\ntrain_1071\ntrain_1072\ntrain_1074\ntrain_1075\ntrain_1076\ntrain_1077\ntrain_1078\ntrain_1079\ntrain_1080\ntrain_1082\ntrain_1083\ntrain_1084\ntrain_1085\ntrain_1086\ntrain_1087\ntrain_1088\ntrain_1090\ntrain_1093\ntrain_1094\ntrain_1095\ntrain_1098\ntrain_1099\ntrain_1100\ntrain_1103\ntrain_1104\ntrain_1105\ntrain_1106\ntrain_1107\ntrain_1110\ntrain_1111\ntrain_1113\ntrain_1114\ntrain_1115\ntrain_1116\ntrain_1119\ntrain_1121\ntrain_1122\ntrain_1125\ntrain_1127\ntrain_1130\ntrain_1131\ntrain_1132\ntrain_1133\ntrain_1134\ntrain_1135\ntrain_1136\ntrain_1138\ntrain_1139\ntrain_1140\ntrain_1141\ntrain_1142\ntrain_1143\ntrain_1144\ntrain_1145\ntrain_1146\ntrain_1149\ntrain_1150\ntrain_1151\ntrain_1153\ntrain_1154\ntrain_1156\ntrain_1157\ntrain_1158\ntrain_1159\ntrain_1161\ntrain_1162\ntrain_1164\ntrain_1165\ntrain_1166\ntrain_1167\ntrain_1168\ntrain_1169\ntrain_1171\ntrain_1172\ntrain_1173\ntrain_1174\ntrain_1176\ntrain_1177\ntrain_1178\ntrain_1179\ntrain_1181\ntrain_1183\ntrain_1184\ntrain_1185\ntrain_1186\ntrain_1187\ntrain_1191\ntrain_1192\ntrain_1193\ntrain_1194\ntrain_1195\ntrain_1196\ntrain_1197\ntrain_1199\ntrain_1200\ntrain_1202\ntrain_1203\ntrain_1204\ntrain_1205\ntrain_1207\ntrain_1208\ntrain_1210\ntrain_1211\ntrain_1212\ntrain_1213\ntrain_1214\ntrain_1217\ntrain_1218\ntrain_1219\ntrain_1221\ntrain_1222\ntrain_1223\ntrain_1224\ntrain_1226\ntrain_1227\ntrain_1228\ntrain_1229\ntrain_1231\ntrain_1236\ntrain_1237\ntrain_1238\ntrain_1240\ntrain_1241\ntrain_1242\ntrain_1244\ntrain_1245\ntrain_1247\ntrain_1250\ntrain_1252\ntrain_1254\ntrain_1255\ntrain_1256\ntrain_1257\ntrain_1259\ntrain_1260\ntrain_1261\ntrain_1262\ntrain_1263\ntrain_1264\ntrain_1266\ntrain_1268\ntrain_1270\ntrain_1271\ntrain_1274\ntrain_1275\ntrain_1276\ntrain_1277\ntrain_1279\ntrain_1280\ntrain_1281\ntrain_1282\ntrain_1283\ntrain_1284\ntrain_1285\ntrain_1286\ntrain_1287\ntrain_1290\ntrain_1291\ntrain_1292\ntrain_1293\ntrain_1294\ntrain_1296\ntrain_1297\ntrain_1299\ntrain_1301\ntrain_1302\ntrain_1303\ntrain_1306\ntrain_1308\ntrain_1312\ntrain_1313\ntrain_1314\ntrain_1315\ntrain_1316\ntrain_1318\ntrain_1319\ntrain_1320\ntrain_1322\ntrain_1323\ntrain_1324\ntrain_1326\ntrain_1327\ntrain_1328\ntrain_1329\ntrain_1330\ntrain_1331\ntrain_1332\ntrain_1333\ntrain_1334\ntrain_1335\ntrain_1336\ntrain_1337\ntrain_1338\ntrain_1339\ntrain_1340\ntrain_1341\ntrain_1343\ntrain_1344\ntrain_1345\ntrain_1347\ntrain_1348\ntrain_1349\ntrain_1354\ntrain_1357\ntrain_1358\ntrain_1359\ntrain_1360\ntrain_1361\ntrain_1362\ntrain_1364\ntrain_1365\ntrain_1366\ntrain_1367\ntrain_1368\ntrain_1369\ntrain_1370\ntrain_1371\ntrain_1372\ntrain_1374\ntrain_1375\ntrain_1376\ntrain_1377\ntrain_1378\ntrain_1381\ntrain_1382\ntrain_1385\ntrain_1386\ntrain_1387\ntrain_1388\ntrain_1389\ntrain_1390\ntrain_1391\ntrain_1392\ntrain_1394\ntrain_1395\ntrain_1396\ntrain_1397\ntrain_1398\ntrain_1400\ntrain_1401\ntrain_1402\ntrain_1403\ntrain_1404\ntrain_1406\ntrain_1407\ntrain_1409\ntrain_1411\ntrain_1412\ntrain_1413\ntrain_1414\ntrain_1415\ntrain_1417\ntrain_1418\ntrain_1419\ntrain_1422\ntrain_1423\ntrain_1424\ntrain_1425\ntrain_1426\ntrain_1427\ntrain_1428\ntrain_1429\ntrain_1430\ntrain_1431\ntrain_1434\ntrain_1435\ntrain_1436\ntrain_1438\ntrain_1439\ntrain_1441\ntrain_1442\ntrain_1445\ntrain_1446\ntrain_1447\ntrain_1449\ntrain_1450\ntrain_1451\ntrain_1452\ntrain_1453\ntrain_1454\ntrain_1457\ntrain_1462\ntrain_1463\ntrain_1464\ntrain_1469\ntrain_1477\ntrain_1479\ntrain_1480\ntrain_1481\ntrain_1482\ntrain_1483\ntrain_1484\ntrain_1485\ntrain_1487\ntrain_1489\ntrain_1490\ntrain_1493\ntrain_1494\ntrain_1495\ntrain_1496\ntrain_1498\ntrain_1499\ntrain_1500\ntrain_1502\ntrain_1503\ntrain_1504\ntrain_1505\ntrain_1507\ntrain_1508\ntrain_1509\ntrain_1510\ntrain_1511\ntrain_1513\ntrain_1514\ntrain_1515\ntrain_1516\ntrain_1517\ntrain_1518\ntrain_1520\ntrain_1521\ntrain_1522\ntrain_1523\ntrain_1524\ntrain_1527\ntrain_1529\ntrain_1531\ntrain_1534\ntrain_1537\ntrain_1538\ntrain_1541\ntrain_1543\ntrain_1545\ntrain_1547\ntrain_1548\ntrain_1549\ntrain_1550\ntrain_1551\ntrain_1552\ntrain_1553\ntrain_1555\ntrain_1558\ntrain_1559\ntrain_1561\ntrain_1562\ntrain_1563\ntrain_1564\ntrain_1565\ntrain_1566\ntrain_1567\ntrain_1568\ntrain_1569\ntrain_1570\ntrain_1573\ntrain_1575\ntrain_1576\ntrain_1577\ntrain_1578\ntrain_1579\ntrain_1580\ntrain_1581\ntrain_1582\ntrain_1583\ntrain_1584\ntrain_1585\ntrain_1587\ntrain_1588\ntrain_1590\ntrain_1591\ntrain_1592\ntrain_1593\ntrain_1594\ntrain_1595\ntrain_1597\ntrain_1599\ntrain_1600\ntrain_1602\ntrain_1603\ntrain_1604\ntrain_1605\ntrain_1607\ntrain_1609\ntrain_1610\ntrain_1611\ntrain_1612\ntrain_1616\ntrain_1618\ntrain_1620\ntrain_1621\ntrain_1622\ntrain_1624\ntrain_1626\ntrain_1628\ntrain_1629\ntrain_1630\ntrain_1631\ntrain_1632\ntrain_1633\ntrain_1634\ntrain_1635\ntrain_1636\ntrain_1638\ntrain_1641\ntrain_1642\ntrain_1643\ntrain_1644\ntrain_1645\ntrain_1646\ntrain_1647\ntrain_1648\ntrain_1649\ntrain_1650\ntrain_1651\ntrain_1652\ntrain_1655\ntrain_1657\ntrain_1659\ntrain_1660\ntrain_1662\ntrain_1663\ntrain_1664\ntrain_1665\ntrain_1667\ntrain_1669\ntrain_1670\ntrain_1672\ntrain_1673\ntrain_1674\ntrain_1675\ntrain_1676\ntrain_1677\ntrain_1678\ntrain_1679\ntrain_1681\ntrain_1682\ntrain_1683\ntrain_1685\ntrain_1686\ntrain_1687\ntrain_1688\ntrain_1689\ntrain_1691\ntrain_1692\ntrain_1694\ntrain_1695\ntrain_1696\ntrain_1698\ntrain_1700\ntrain_1701\ntrain_1704\ntrain_1706\ntrain_1709\ntrain_1711\ntrain_1714\ntrain_1716\ntrain_1717\ntrain_1720\ntrain_1721\ntrain_1722\ntrain_1723\ntrain_1725\ntrain_1728\ntrain_1729\ntrain_1730\ntrain_1733\ntrain_1734\ntrain_1735\ntrain_1736\ntrain_1738\ntrain_1739\ntrain_1740\ntrain_1741\ntrain_1742\ntrain_1743\ntrain_1748\ntrain_1749\ntrain_1750\ntrain_1751\ntrain_1753\n', 'adapter_valid_ids.txt': 'train_0001\ntrain_0004\ntrain_0015\ntrain_0016\ntrain_0017\ntrain_0031\ntrain_0063\ntrain_0066\ntrain_0075\ntrain_0080\ntrain_0088\ntrain_0090\ntrain_0095\ntrain_0106\ntrain_0110\ntrain_0117\ntrain_0130\ntrain_0133\ntrain_0141\ntrain_0144\ntrain_0145\ntrain_0147\ntrain_0162\ntrain_0163\ntrain_0164\ntrain_0169\ntrain_0172\ntrain_0181\ntrain_0194\ntrain_0196\ntrain_0203\ntrain_0207\ntrain_0214\ntrain_0216\ntrain_0219\ntrain_0225\ntrain_0228\ntrain_0231\ntrain_0233\ntrain_0236\ntrain_0240\ntrain_0255\ntrain_0278\ntrain_0285\ntrain_0304\ntrain_0313\ntrain_0316\ntrain_0321\ntrain_0330\ntrain_0344\ntrain_0345\ntrain_0346\ntrain_0350\ntrain_0354\ntrain_0357\ntrain_0378\ntrain_0381\ntrain_0395\ntrain_0405\ntrain_0422\ntrain_0435\ntrain_0442\ntrain_0453\ntrain_0457\ntrain_0468\ntrain_0469\ntrain_0476\ntrain_0480\ntrain_0481\ntrain_0486\ntrain_0497\ntrain_0500\ntrain_0505\ntrain_0508\ntrain_0518\ntrain_0536\ntrain_0538\ntrain_0550\ntrain_0554\ntrain_0560\ntrain_0563\ntrain_0572\ntrain_0577\ntrain_0596\ntrain_0598\ntrain_0618\ntrain_0621\ntrain_0624\ntrain_0625\ntrain_0628\ntrain_0664\ntrain_0689\ntrain_0694\ntrain_0719\ntrain_0729\ntrain_0752\ntrain_0764\ntrain_0770\ntrain_0772\ntrain_0777\ntrain_0779\ntrain_0787\ntrain_0790\ntrain_0800\ntrain_0805\ntrain_0814\ntrain_0815\ntrain_0819\ntrain_0820\ntrain_0834\ntrain_0857\ntrain_0864\ntrain_0867\ntrain_0870\ntrain_0875\ntrain_0923\ntrain_0933\ntrain_0935\ntrain_0943\ntrain_0945\ntrain_0947\ntrain_0948\ntrain_0953\ntrain_0958\ntrain_0959\ntrain_0960\ntrain_0977\ntrain_0978\ntrain_0991\ntrain_1000\ntrain_1014\ntrain_1016\ntrain_1019\ntrain_1021\ntrain_1049\ntrain_1060\ntrain_1062\ntrain_1070\ntrain_1081\ntrain_1089\ntrain_1091\ntrain_1096\ntrain_1097\ntrain_1109\ntrain_1117\ntrain_1118\ntrain_1123\ntrain_1124\ntrain_1128\ntrain_1137\ntrain_1147\ntrain_1148\ntrain_1155\ntrain_1160\ntrain_1163\ntrain_1175\ntrain_1180\ntrain_1188\ntrain_1201\ntrain_1206\ntrain_1209\ntrain_1215\ntrain_1216\ntrain_1233\ntrain_1234\ntrain_1246\ntrain_1248\ntrain_1249\ntrain_1253\ntrain_1265\ntrain_1278\ntrain_1289\ntrain_1295\ntrain_1300\ntrain_1305\ntrain_1310\ntrain_1311\ntrain_1317\ntrain_1325\ntrain_1346\ntrain_1351\ntrain_1352\ntrain_1353\ntrain_1355\ntrain_1356\ntrain_1363\ntrain_1373\ntrain_1379\ntrain_1380\ntrain_1384\ntrain_1399\ntrain_1405\ntrain_1408\ntrain_1432\ntrain_1433\ntrain_1437\ntrain_1440\ntrain_1448\ntrain_1458\ntrain_1460\ntrain_1471\ntrain_1472\ntrain_1474\ntrain_1475\ntrain_1476\ntrain_1478\ntrain_1486\ntrain_1491\ntrain_1497\ntrain_1506\ntrain_1512\ntrain_1519\ntrain_1526\ntrain_1528\ntrain_1533\ntrain_1540\ntrain_1542\ntrain_1544\ntrain_1546\ntrain_1556\ntrain_1557\ntrain_1560\ntrain_1571\ntrain_1572\ntrain_1589\ntrain_1598\ntrain_1606\ntrain_1608\ntrain_1615\ntrain_1617\ntrain_1623\ntrain_1653\ntrain_1654\ntrain_1658\ntrain_1661\ntrain_1668\ntrain_1697\ntrain_1702\ntrain_1703\ntrain_1705\ntrain_1707\ntrain_1712\ntrain_1713\ntrain_1715\ntrain_1746\n', 'requirements_asr_2gpu.lock': 'cffi==2.1.1\nnumpy==2.2.6\npycparser==3.12\nscipy==1.16.2\nsherpa-onnx==2.0.1\nsherpa-onnx-core==2.0.1\nsoundfile==0.13.1\n', 'requirements_infer_2gpu.lock': '--extra-index-url https://download.pytorch.org/whl/cu128\naccelerate==1.15.0\nagent-detector==2.0.0\naiohappyeyeballs==2.7.1\naiohttp==3.14.5\naiosignal==1.4.0\nairportsdata==20261030\nannotated-doc==0.0.5\nannotated-types==0.8.0\nanyio==4.16.0\nastor==0.8.1\nattrs==26.1.0\nblake3==1.0.11\ncachetools==7.2.8\ncertifi==2026.10.23\ncharset-normalizer==3.5.2\nclick==8.5.0\ncloudpickle==3.1.2\ncompressed-tensors==0.10.2\ncuda-pathfinder==1.8.5\ncupy-cuda12x==14.2.0\ndepyf==0.18.0\ndetect-installer==0.2.1\ndill==0.4.1\ndiskcache==5.6.3\ndistro==1.9.0\ndnspython==2.9.1\neinops==0.8.2\nemail-validator==2.3.0\nfastapi==0.162.0\nfastapi-cli==0.0.32\nfastapi-cloud-cli==0.26.0\nfastar==0.12.0\nfilelock==4.2.7\nfrozenlist==1.8.0\nfsspec==2026.9.0\ngguf==0.19.0\ngoogleapis-common-protos==1.76.0\nh11==0.16.0\nhf-xet==1.8.0\nhttpcore==1.0.9\nhttpcore2==2.16.0\nhttptools==0.9.0\nhttpx==0.28.1\nhttpx2==2.16.0\nhuggingface-hub==0.35.3\nidna==3.20\ninteregular==0.3.3\njinja2==3.1.6\njiter==0.18.0\njsonschema==4.25.1\njsonschema-specifications==2025.9.1\nlark==1.2.2\nllguidance==0.7.30\nllvmlite==0.44.0\nlm-format-enforcer==0.10.12\nmarkdown-it-py==4.2.0\nmarkupsafe==3.0.4\nmdurl==0.1.2\nmistral-common==1.13.1\nmpmath==1.3.0\nmsgpack==1.2.4\nmsgspec==0.22.0\nmultidict==7.2.1\nnest-asyncio==1.6.0\nnetworkx==3.7\nninja==1.13.2\nnumba==0.61.2\nnumpy==2.2.6\nnvidia-cublas-cu12==12.8.3.14\nnvidia-cuda-cupti-cu12==12.8.57\nnvidia-cuda-nvrtc-cu12==12.8.61\nnvidia-cuda-runtime-cu12==12.8.57\nnvidia-cudnn-cu12==9.7.1.26\nnvidia-cufft-cu12==11.3.3.41\nnvidia-cufile-cu12==1.13.0.11\nnvidia-curand-cu12==10.3.9.55\nnvidia-cusolver-cu12==11.7.2.55\nnvidia-cusparse-cu12==12.5.7.53\nnvidia-cusparselt-cu12==0.6.3\nnvidia-nccl-cu12==2.26.2\nnvidia-nvjitlink-cu12==12.8.61\nnvidia-nvtx-cu12==12.8.55\nopenai==1.90.0\nopencv-python-headless==5.0.0.93\nopentelemetry-api==1.45.2\nopentelemetry-exporter-http-transport==0.66b2\nopentelemetry-exporter-otlp-common==0.66b2\nopentelemetry-exporter-otlp-proto-common==1.45.2\nopentelemetry-exporter-otlp-proto-http==1.45.2\nopentelemetry-proto==1.45.2\nopentelemetry-sdk==1.45.2\nopentelemetry-semantic-conventions==0.66b2\noutlines==0.1.11\noutlines-core==0.1.26\npackaging==26.3\npartial-json-parser==0.2.1.1.post7\npeft==0.17.1\npillow==13.0.0\nprometheus-client==0.26.0\nprometheus-fastapi-instrumentator==8.1.0\npropcache==0.5.4\nprotobuf==7.36.2\npsutil==7.2.2\npy-cpuinfo==9.0.0\npybase64==1.5.1\npycountry==26.2.16\npydantic==2.14.2\npydantic-core==2.50.2\npydantic-extra-types==2.11.1\npydantic-settings==2.16.2\npygments==2.21.0\npython-dotenv==1.2.4\npython-json-logger==4.2.0\npython-multipart==0.0.33\npyyaml==6.0.3\npyzmq==27.2.0\nray==2.59.0\nreferencing==0.37.0\nregex==2026.9.29\nrequests==2.34.2\nrich==15.0.0\nrich-toolkit==0.20.6\nrignore==0.8.1\nrpds-py==2026.9.1\nsafetensors==0.8.0\nscipy==1.18.1\nsentencepiece==0.2.2\nsentry-sdk==2.73.0\nsetuptools==79.0.1\nshellingham==1.5.4\nsix==1.17.0\nsniffio==1.3.1\nstarlette==1.10.0\nsympy==1.14.0\ntiktoken==0.14.0\ntokenizers==0.21.4\ntorch==2.7.0+cu128\ntorchaudio==2.7.0+cu128\ntorchvision==0.22.0+cu128\ntqdm==4.70.1\ntransformers==4.53.3\ntriton==3.3.0\ntruststore==0.10.5\ntyper==0.27.3\ntyping-extensions==4.16.0\ntyping-inspection==0.4.4\nurllib3==2.8.0\nuvicorn==0.54.0\nuvloop==0.23.0\nvllm==0.9.2\nwatchfiles==1.3.0\nwebsockets==17.4\nwrapt==1.17.3\nxformers==0.0.30\nxgrammar==0.1.19\nyarl==1.25.1\n', 'split_200.txt': 'train_0006\ntrain_0014\ntrain_0030\ntrain_0045\ntrain_0050\ntrain_0057\ntrain_0065\ntrain_0070\ntrain_0094\ntrain_0111\ntrain_0121\ntrain_0130\ntrain_0135\ntrain_0136\ntrain_0140\ntrain_0149\ntrain_0155\ntrain_0172\ntrain_0179\ntrain_0189\ntrain_0193\ntrain_0195\ntrain_0205\ntrain_0210\ntrain_0219\ntrain_0228\ntrain_0229\ntrain_0231\ntrain_0234\ntrain_0239\ntrain_0248\ntrain_0252\ntrain_0258\ntrain_0262\ntrain_0266\ntrain_0286\ntrain_0325\ntrain_0340\ntrain_0373\ntrain_0398\ntrain_0401\ntrain_0410\ntrain_0415\ntrain_0440\ntrain_0483\ntrain_0526\ntrain_0532\ntrain_0536\ntrain_0540\ntrain_0554\ntrain_0605\ntrain_0626\ntrain_0632\ntrain_0648\ntrain_0651\ntrain_0657\ntrain_0677\ntrain_0694\ntrain_0700\ntrain_0709\ntrain_0710\ntrain_0715\ntrain_0716\ntrain_0738\ntrain_0740\ntrain_0748\ntrain_0755\ntrain_0773\ntrain_0777\ntrain_0779\ntrain_0783\ntrain_0788\ntrain_0789\ntrain_0792\ntrain_0793\ntrain_0794\ntrain_0802\ntrain_0805\ntrain_0814\ntrain_0827\ntrain_0829\ntrain_0830\ntrain_0839\ntrain_0843\ntrain_0846\ntrain_0849\ntrain_0850\ntrain_0863\ntrain_0868\ntrain_0871\ntrain_0886\ntrain_0889\ntrain_0890\ntrain_0893\ntrain_0911\ntrain_0920\ntrain_0925\ntrain_0930\ntrain_0933\ntrain_0934\ntrain_0944\ntrain_0945\ntrain_0960\ntrain_0962\ntrain_0968\ntrain_0970\ntrain_0992\ntrain_0996\ntrain_0999\ntrain_1006\ntrain_1009\ntrain_1011\ntrain_1012\ntrain_1015\ntrain_1022\ntrain_1025\ntrain_1031\ntrain_1034\ntrain_1050\ntrain_1057\ntrain_1061\ntrain_1062\ntrain_1064\ntrain_1082\ntrain_1095\ntrain_1096\ntrain_1100\ntrain_1104\ntrain_1107\ntrain_1115\ntrain_1149\ntrain_1158\ntrain_1163\ntrain_1177\ntrain_1181\ntrain_1186\ntrain_1187\ntrain_1188\ntrain_1196\ntrain_1197\ntrain_1204\ntrain_1206\ntrain_1224\ntrain_1233\ntrain_1234\ntrain_1263\ntrain_1274\ntrain_1280\ntrain_1284\ntrain_1293\ntrain_1295\ntrain_1306\ntrain_1314\ntrain_1316\ntrain_1335\ntrain_1339\ntrain_1340\ntrain_1349\ntrain_1389\ntrain_1394\ntrain_1397\ntrain_1399\ntrain_1401\ntrain_1405\ntrain_1419\ntrain_1425\ntrain_1426\ntrain_1428\ntrain_1436\ntrain_1441\ntrain_1450\ntrain_1463\ntrain_1474\ntrain_1476\ntrain_1495\ntrain_1504\ntrain_1524\ntrain_1526\ntrain_1531\ntrain_1546\ntrain_1555\ntrain_1561\ntrain_1592\ntrain_1594\ntrain_1600\ntrain_1622\ntrain_1630\ntrain_1633\ntrain_1635\ntrain_1649\ntrain_1652\ntrain_1659\ntrain_1685\ntrain_1700\ntrain_1705\ntrain_1713\ntrain_1715\ntrain_1716\ntrain_1721\ntrain_1733\n', 'umilife/infer_2gpu.py': '"""Fine-tuned multi-LoRA workflow; held-out labels are never model inputs."""\nimport os\nos.environ.setdefault("CUDA_VISIBLE_DEVICES","0")\nos.environ.setdefault("VLLM_USE_V1","0")\nos.environ.setdefault("VLLM_ATTENTION_BACKEND","XFORMERS")\nos.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD","spawn")\nos.environ.setdefault("VLLM_USAGE_STATS_ENABLED","0")\nos.environ.setdefault("DO_NOT_TRACK","1")\nimport json,time,argparse,math\nfrom pathlib import Path\nfrom collections import Counter\nfrom umilife.core import read_jsonl,write_json,dumps,parse_json,allowed_specs,load_catalog,import_toolenv\nfrom umilife.prompt_policy import (PROMPTS,COMPLETION_REVIEW,POLICY_AUDIT,POLICY_VERSION,\n    build_messages,request_schema,validate_meta_action,repeated_read,canonical,digest)\nfrom copy import deepcopy\nSYSTEMS=PROMPTS\nfrom umilife.data import normalized_row\nfrom umilife.grammar import role_schema\n\ndef rollout(row,text,specs,env,mode,max_calls=7,max_decisions=7):\n    assert mode==\'multi\',\'This evaluator uses two local adapters only\'\n    started=time.monotonic();trajectory=[];trace=[];answer=None;reason=\'decision_limit\'\n    feedback=None;reviews=0;read_skips=0;repairs=0\n    revisions={\'planner\':0,\'caller\':0}\n    def generate(role,decision,plan=None,feedback=None):\n        response=yield {\'role\':role,\'messages\':build_messages(role,row,text,specs,trajectory,plan,feedback),\n                        \'schema\':request_schema(role,specs,plan)}\n        trace.append({\'role\':role,\'decision\':decision,**response})\n        if response.get(\'schema_valid\') is False:raise ValueError(\'Invalid role schema: \'+response.get(\'schema_error\',\'\'))\n        result=parse_json(response[\'raw\'])\n        if role==\'planner\':\n            if not isinstance(result,dict) or result.get(\'next\') not in (\'caller\',\'summarizer\',\'give_up\'):raise ValueError(\'Invalid Planner route\')\n            if result[\'next\']==\'caller\' and result.get(\'tool\') not in specs:raise ValueError(\'Unavailable tool\')\n        return result\n    for decision in range(max_decisions):\n        try:\n            # One initial plan plus ONE shared revision slot, for repair OR ending review.\n            plan=None;planner_calls=0;planner_feedback=feedback;feedback=None\n            while planner_calls<2:\n                planner_calls+=1\n                if planner_calls==2:revisions[\'planner\']+=1\n                try:\n                    plan=yield from generate(\'planner\',decision,feedback=planner_feedback)\n                    trace.append({\'role\':\'planner\',\'decision\':decision,\'output\':plan})\n                    if plan[\'next\'] in (\'summarizer\',\'give_up\') and planner_calls==1:\n                        reviews+=1\n                        planner_feedback={\'kind\':\'completion_review\',\'proposal\':plan,\'instruction\':COMPLETION_REVIEW}\n                        continue\n                    break\n                except (ValueError,KeyError,TypeError) as exc:\n                    trace.append({\'role\':\'planner\',\'decision\':decision,\'error\':str(exc)})\n                    if planner_calls==2:raise\n                    repairs+=1;planner_feedback={\'kind\':\'planning_contract_error\',\'error\':str(exc),\n                        \'instruction\':\'Plan one valid step using known evidence and installed tools.\'}\n            route=plan[\'next\']\n            if route==\'give_up\':reason=\'give_up\';break\n            if route==\'summarizer\':reason=\'complete\';break\n            if len(env.get_tool_calls())>=max_calls:reason=\'call_limit\';break\n            action=None;action_feedback=None\n            for attempt in range(2):\n                if attempt:revisions[\'caller\']+=1;repairs+=1\n                try:\n                    candidate=yield from generate(\'caller\',decision,plan,action_feedback)\n                    if not isinstance(candidate,dict) or candidate.get(\'name\')!=plan[\'tool\']:raise ValueError(\'Caller must use selected tool\')\n                    check=env.validate(candidate[\'name\'],candidate.get(\'arguments\'))\n                    if check.get(\'ok\'):\n                        extra=validate_meta_action(candidate,specs)\n                        if extra:check={\'ok\':False,\'error\':extra}\n                    trace.append({\'role\':\'caller\',\'decision\':decision,\'output\':candidate,\'validation\':check})\n                    if check.get(\'ok\'):action=candidate;break\n                    error=str(check.get(\'error\',\'Invalid arguments\'))\n                except (ValueError,KeyError,TypeError) as exc:error=str(exc)\n                action_feedback={\'kind\':\'tool_contract_error\',\'error\':error,\n                    \'instruction\':\'Correct the selected call using known evidence; do not invent required values.\'}\n            if action is None:\n                reason=\'format_error\';trace.append({\'error\':\'Caller exhausted one allowed revision\',\'decision\':decision});break\n            if repeated_read(action,specs,trajectory):\n                read_skips+=1\n                feedback={\'kind\':\'repeated_read\',\'action\':action,\n                    \'instruction\':\'Reuse the unchanged result; change strategy or ask for missing information instead of repeating it.\'}\n                trace.append({\'event\':\'repeat_read_prevented\',\'decision\':decision,\'action\':action});continue\n            observed=env.call(action[\'name\'],action[\'arguments\'])\n            trajectory.append({\'step\':len(trajectory),\'thought\':plan.get(\'thought\',\'\'),\'action\':action,\'observation\':observed})\n            if action[\'name\']==\'sys.ask_missing_info\':reason=\'needs_user_input\';break\n        except (ValueError,KeyError,TypeError) as exc:\n            reason=\'format_error\';trace.append({\'error\':str(exc),\'decision\':decision});break\n    return {\'id\':row[\'id\'],\'tool_calls\':env.get_tool_calls(),\'state_hash\':env.get_state_hash()},{\n        \'id\':row[\'id\'],\'reason\':reason,\'elapsed_s\':time.monotonic()-started,\'over_deadline\':False,\n        \'time_limit_enforced\':False,\'answer\':answer,\'trace\':trace,\'trajectory\':trajectory,\'transcript\':text,\n        \'completion_reviews\':reviews,\'repeat_reads_prevented\':read_skips,\'repairs\':repairs,\n        \'revision_counts\':revisions,\'max_revisions_per_role_per_decision\':1,\'max_decisions\':max_decisions,\'max_tool_calls\':max_calls,\'prompt_policy\':POLICY_VERSION}\n\ndef run_batch(generators,backend):\n    active={}\n    completed={}\n    for index,generator in enumerate(generators):\n        try:active[index]=next(generator)\n        except StopIteration as stop:completed[index]=stop.value\n    while active:\n        good=[];bad=[]\n        for index,request in active.items():\n            try:backend.validate(request);good.append(index)\n            except ValueError as exc:bad.append((index,exc))\n        for index,error in bad:\n            try:active[index]=generators[index].throw(error)\n            except StopIteration as stop:completed[index]=stop.value;del active[index]\n        if not good:continue\n        responses=backend.generate([active[index] for index in good])\n        assert len(responses)==len(good)\n        for index,response in zip(good,responses,strict=True):\n            try:active[index]=generators[index].send(response)\n            except StopIteration as stop:completed[index]=stop.value;del active[index]\n    return [completed[index] for index in range(len(generators))]\n\ndef prepare_t4_lora_kernel():\n    """Work around Triton 3.3 fp16 conversion failure on SM75; preserve FP32 accumulation."""\n    import torch,importlib.util,importlib.metadata,hashlib\n    path=Path(importlib.util.find_spec("vllm").origin).parent/"lora/ops/triton_ops/kernel_utils.py"\n    original="        accumulator += tl.dot(\\n            tiled_a,\\n            tiled_b,\\n        )"\n    patched="        # LifeMate SM75 compatibility: explicit exact fp16->fp32 before dot.\\n        accumulator += tl.dot(tiled_a.to(tl.float32), tiled_b.to(tl.float32), input_precision=\\"ieee\\")"\n    source=path.read_text()\n    capability=torch.cuda.get_device_capability(0)\n    if capability[0]>=8:\n        if patched in source:path.write_text(source.replace(patched,original))\n        return {"applied":False,"capability":list(capability)}\n    assert importlib.metadata.version("vllm")=="0.9.2"\n    assert importlib.metadata.version("triton")=="3.3.0"\n    assert original in source or patched in source,"Unexpected LoRA kernel source"\n    if original in source:\n        import tempfile\n        with tempfile.NamedTemporaryFile(mode=\'w\',dir=path.parent,delete=False,suffix=\'.py\') as tmp:\n            tmp.write(source.replace(original,patched));temporary=tmp.name\n        os.replace(temporary,path)\n    # Exercise the exact patched helper on the GPU against an FP32 PyTorch reference.\n    global triton,tl,mm_k\n    import triton\n    import triton.language as tl\n    from vllm.lora.ops.triton_ops.kernel_utils import mm_k\n    @triton.jit\n    def check_kernel(A,B,C,K:tl.constexpr):\n        rows=tl.arange(0,32)\n        cols=tl.arange(0,16)\n        kk=tl.arange(0,32)\n        a=A+rows[:,None]*K+kk[None,:]\n        b=B+kk[:,None]*16+cols[None,:]\n        acc=mm_k(a,b,1,16,kk,K,32,16,32,True,1,False,tl.float16)\n        tl.store(C+rows[:,None]*16+cols[None,:],acc)\n    errors=[]\n    torch.manual_seed(42)\n    for k in (128,2560):\n        a=torch.randn(32,k,device="cuda",dtype=torch.float16)\n        b=torch.randn(k,16,device="cuda",dtype=torch.float16)\n        c=torch.empty(32,16,device="cuda",dtype=torch.float32)\n        check_kernel[(1,)](a,b,c,k,num_warps=4)\n        ref=a.float()@b.float()\n        torch.testing.assert_close(c,ref,rtol=5e-4,atol=2e-3)\n        errors.append(float((c-ref).abs().max()))\n    return {"applied":True,"capability":list(capability),"operation":"LoRA dot with FP32 accumulation",\n            "reference_test":"pass","maximum_errors":errors,"kernel_sha256":hashlib.sha256(path.read_bytes()).hexdigest()}\n\nclass Backend:\n    def __init__(self,model,revision,adapters,max_context=8192):\n        self.t4_compatibility=prepare_t4_lora_kernel()\n        print(dumps({"t4_compatibility":self.t4_compatibility}),flush=True)\n        from vllm import LLM,SamplingParams\n        from vllm.lora.request import LoRARequest\n        from transformers import AutoConfig,AutoModelForCausalLM\n        from safetensors import safe_open\n        import torch\n        assert torch.cuda.is_available()\n        assert torch.cuda.get_device_capability(0)>=(7,0)\n        from umilife.runtime_options import engine_options\n        self.runtime_options=engine_options(torch.cuda.get_device_capability(0))\n        print(dumps({\'event\':\'runtime_options\',\'gpu\':os.environ.get(\'LIFEMATE_WORKER_GPU\',\'0\'),**self.runtime_options}),flush=True)\n        self.max_context=max_context\n        self.requests={}\n        adapter_parameters=0\n        lineage_records={}\n        for index,role in enumerate(("planner","caller"),start=1):\n            path=Path(adapters)/role\n            meta=json.loads((path/"lineage.json").read_text())\n            assert meta["stage"]==role and meta["model"]==model and meta["revision"]==revision\n            assert meta["final_adapter"]["finite"]\n            assert meta[\'prompt_sha256\']==digest(PROMPTS),\'Training/inference policy mismatch\'\n            assert meta[\'world_size\']==2 and meta[\'micro_batch\']>0 and meta[\'eval_batch_per_device\']>0\n            lineage_records[role]=meta\n            self.requests[role]=LoRARequest(role,index,str(path))\n            with safe_open(path/"adapter_model.safetensors",framework="pt",device="cpu") as handle:\n                adapter_parameters+=sum(math.prod(handle.get_slice(k).get_shape()) for k in handle.keys())\n        manifest_hashes={v[\'dataset_manifest_sha256\'] for v in lineage_records.values()}\n        # Raw JSON manifest hashes can differ solely because key insertion order differs.\n        contract=json.loads((Path(adapters)/"data_contract.json").read_text())\n        assert {role:meta["dataset_manifest_sha256"] for role,meta in lineage_records.items()}==contract["raw_manifest_sha256"]\n        self.dataset_manifest_sha256=contract["canonical_manifest_sha256"]\n        config=AutoConfig.from_pretrained(model,revision=revision)\n        with torch.device("meta"):meta_model=AutoModelForCausalLM.from_config(config)\n        base_parameters=meta_model.num_parameters();del meta_model\n        self.parameter_report={"base":base_parameters,"comparison_adapters":adapter_parameters,\n                               "total_llm":base_parameters+adapter_parameters}\n        assert self.parameter_report["total_llm"]<5_000_000_000\n        self.llm=LLM(model=model,revision=revision,tokenizer=str(Path(adapters)/"planner"),\n            dtype="float16",tensor_parallel_size=1,enable_lora=True,max_loras=2,max_cpu_loras=2,\n            max_lora_rank=16,max_model_len=max_context,max_num_seqs=4,max_num_batched_tokens=max_context,\n            gpu_memory_utilization=.85,enforce_eager=not self.runtime_options["cuda_graphs"],enable_prefix_caching=self.runtime_options["prefix_caching"],\n            enable_chunked_prefill=False,seed=42,trust_remote_code=False,\n            generation_config="vllm",disable_log_stats=True)\n        self.tokenizer=self.llm.get_tokenizer()\n        self.max_tokens={"global":768,"planner":320,"caller":512,"summarizer":384}\n        self.batch_sizes=[];self.roles_seen=Counter();self.schema_violations=0\n    def make_sampling(self,request):\n        from vllm import SamplingParams\n        # No token-level grammar or logits processor, for either local adapter.\n        return SamplingParams(temperature=0,seed=42,\n            max_tokens=self.max_tokens[request["role"]],guided_decoding=None)\n    def validate(self,request):\n        role=request["role"]\n        prompt=self.tokenizer.apply_chat_template(request["messages"],tokenize=False,add_generation_prompt=True)\n        length=len(self.tokenizer.encode(prompt,add_special_tokens=False))\n        if length+(64 if request.get("probe") else self.max_tokens[role])>self.max_context:\n            raise ValueError("Context budget exceeded; no evidence is truncated")\n    def generate(self,requests):\n        assert 1<=len(requests)<=4\n        for r in requests:self.validate(r)\n        self.batch_sizes.append(len(requests));self.roles_seen.update(r["role"] for r in requests)\n        prompts=[self.tokenizer.apply_chat_template(r["messages"],tokenize=False,add_generation_prompt=True) for r in requests]\n        generation_started=time.monotonic()\n        results=self.llm.generate(prompts,[self.make_sampling(r) for r in requests],\n            lora_request=[self.requests[r["role"]] for r in requests],use_tqdm=False)\n        from jsonschema import Draft202012Validator,ValidationError\n        responses=[]\n        for request,result in zip(requests,results,strict=True):\n            response={"raw":result.outputs[0].text,"guided_decoding":False,"finish_reason":result.outputs[0].finish_reason,\n                      "stop_reason":result.outputs[0].stop_reason,\n                      "prompt_sha256":digest(request["messages"]),"schema_sha256":digest(request.get("schema",role_schema(request["role"])))}\n            try:\n                value=parse_json(response["raw"])\n                Draft202012Validator(request.get("schema",role_schema(request["role"]))).validate(value)\n                response["schema_valid"]=True\n            except (ValueError,ValidationError) as exc:\n                response.update(schema_valid=False,schema_error=str(exc)[:300])\n                self.schema_violations+=1\n            responses.append(response)\n        print(dumps({\'event\':\'generation\',\'gpu\':os.environ.get(\'LIFEMATE_WORKER_GPU\',\'0\'),\'roles\':dict(Counter(r[\'role\'] for r in requests)),\'batch\':len(requests),\'generated_tokens\':sum(len(r.outputs[0].token_ids) for r in results),\'seconds\':time.monotonic()-generation_started}),flush=True)\n        return responses\n    def preflight(self):\n        # The original probes relied on LMFE to impose a synthetic output schema.\n        # Skip those probes: real outputs still undergo JSON/schema validation.\n        return {\'roles\':[\'planner\',\'caller\'],\'batch_limit\':4,\'guided_decoding\':False,\n                \'synthetic_json_probes\':\'skipped_requires_grammar\',\n                \'post_generation_schema_validation\':True,\'runtime\':self.runtime_options}\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'data-root\',\'work\',\'model\',\'revision\',\'adapters\',\'ids-file\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--deadline-monotonic\',type=float)\n    p.add_argument(\'--split\',choices=[\'validation\',\'public_test\'],required=True)\n    p.add_argument(\'--mode\',choices=[\'single\',\'multi_local\'],default=\'multi_local\')\n    a=p.parse_args();root=Path(a.data_root);work=Path(a.work);work.mkdir(parents=True,exist_ok=True)\n    split=\'train\' if a.split==\'validation\' else \'public_test\'\n    ids=Path(a.ids_file).read_text().splitlines();assert len(ids)==len(set(ids)) and ids\n    source=read_jsonl(root/split/f\'{split}.jsonl\');lookup={r[\'id\']:r for r in source};assert set(ids)<=lookup.keys()\n    if a.split==\'public_test\':assert set(ids)==set(lookup),\'Test must include every public-test ID\'\n    rows=[]\n    for sid in ids:\n        raw=lookup[sid]\n        context={k:deepcopy(raw[\'context\'][k]) for k in (\'current_time\',\'timezone\',\'location\',\'user_profile\') if k in raw[\'context\']}\n        context[\'installed_apps\']={app:{\'tools\':list(meta.get(\'tools\',[]))} for app,meta in raw[\'context\'][\'installed_apps\'].items()}\n        history=[]\n        for turn in raw.get(\'history\',[]):\n            keys=(\'role\',\'audio\') if turn[\'role\']==\'user\' else (\'role\',\'text\',\'tool_calls\',\'tool_results\')\n            history.append({k:deepcopy(turn[k]) for k in keys if k in turn})\n        rows.append({\'id\':sid,\'audio\':raw[\'audio\'],\'context\':context,\'history\':history})\n    del source,lookup\n    transcripts=json.loads((work/\'asr_transcripts.json\').read_text());asr=json.loads((work/\'asr_report.json\').read_text())\n    assert not asr[\'oracle_fallback\'] and asr[\'validation_samples\']>=len(rows)\n    assert all(raw[\'audio\'] in transcripts for raw in rows)\n    backend=Backend(a.model,a.revision,a.adapters);preflight=backend.preflight();write_json(work/\'preflight.json\',preflight)\n    catalog=load_catalog(root,split);ToolEnv=import_toolenv(root,split)\n    batches=work/\'batches\';batches.mkdir(exist_ok=True)\n    active=work/\'active_batch.json\'\n    if active.exists():\n        marker=json.loads(active.read_text())\n        assert (batches/marker[\'file\']).exists(),\'An interrupted batch has unrecorded tool execution. Do not silently replay it.\'\n        active.unlink()\n    predictions=[];traces=[];mode=\'single\' if a.mode==\'single\' else \'multi\'\n    with (work/\'submission.jsonl\').open(\'w\') as pf,(work/\'traces.jsonl\').open(\'w\') as tf:\n        for start in range(0,len(rows),4):\n            if a.deadline_monotonic and time.monotonic()>a.deadline_monotonic and not (batches/f\'{start:06}.json\').exists():\n                write_json(work/\'progress.tmp\',{\'paused\':True,\'completed\':len(predictions),\'total\':len(rows),\'next_index\':start})\n                (work/\'progress.tmp\').replace(work/\'progress.json\')\n                print(\'PAUSED_AT_BATCH_BOUNDARY: preserve this run directory and resume in another session\',flush=True)\n                raise SystemExit(75)\n            group=rows[start:start+4]\n            batch_file=batches/f\'{start:06}.json\'\n            if batch_file.exists():\n                completed=json.loads(batch_file.read_text())\n                assert [p[0][\'id\'] for p in completed]==[r[\'id\'] for r in group]\n            else:\n                write_json(active,{\'file\':batch_file.name,\'ids\':[r[\'id\'] for r in group]})\n            if not batch_file.exists():\n                envs=[ToolEnv(split,raw[\'id\'],root=root) for raw in group]\n                safe_rows=[normalized_row(raw,transcript=transcripts) for raw in group]\n                gens=[rollout(row,transcripts[raw[\'audio\']],allowed_specs(row,catalog),env,mode)\n                      for raw,row,env in zip(group,safe_rows,envs,strict=True)]\n                completed=run_batch(gens,backend)\n                temp=batch_file.with_suffix(\'.tmp\');write_json(temp,completed);temp.replace(batch_file);active.unlink()\n            for pred,trace in completed:\n                predictions.append(pred);traces.append(trace);pf.write(dumps(pred)+\'\\n\');pf.flush();tf.write(dumps(trace)+\'\\n\');tf.flush()\n            write_json(work/\'progress.tmp\',{\'completed\':len(predictions),\'total\':len(rows)})\n            (work/\'progress.tmp\').replace(work/\'progress.json\')\n            print(dumps({\'split\':a.split,\'mode\':a.mode,\'completed\':len(predictions),\'total\':len(rows)}),flush=True)\n    assert [r[\'id\'] for r in predictions]==ids\n    report={\'split\':a.split,\'mode\':a.mode,\'samples\':len(rows),\'asr\':asr,\'preflight\':preflight,\n            \'actual_batch_sizes\':dict(Counter(backend.batch_sizes)),\'schema_violations\':backend.schema_violations,\n            \'role_adapter_map\':{\'planner\':\'planner\',\'caller\':\'caller\',\'summarizer\':\'terminal_without_generation\'},\n            \'missing_local_adapters\':[],\'policy\':POLICY_AUDIT,\n            \'termination_reasons\':dict(Counter(t[\'reason\'] for t in traces)),\n            \'guided_decoding\':False,\'post_generation_schema_validation\':True,\n            \'runtime\':{**backend.runtime_options,\'worker_gpu\':os.environ.get(\'LIFEMATE_WORKER_GPU\',\'0\'),\'batch_per_gpu\':4,\'max_revisions_per_role_per_decision\':1}}\n    if a.split==\'validation\':\n        # Scorer-only gold replay after ALL model predictions have been generated.\n        from umilife.evaluate import evaluate\n        references=[]\n        for raw in read_jsonl(root/\'train/train.jsonl\'):\n            if raw[\'id\'] not in set(ids):continue\n            env=ToolEnv(\'train\',raw[\'id\'],root=root)\n            for action in raw[\'label\'][\'tool_calls\']:env.call(action[\'name\'],action[\'arguments\'])\n            references.append({\'id\':raw[\'id\'],\'tool_calls\':raw[\'label\'][\'tool_calls\'],\'state_hash\':env.get_state_hash()})\n        report[\'metrics\']=evaluate(predictions,references,catalog,traces)\n    write_json(work/\'report.json\',report)\n    print(dumps({k:v for k,v in report.items() if k!=\'metrics\'}),flush=True)\nif __name__==\'__main__\':main()\n', 'umilife/parallel_validation.py': '"""Two independent GPU workers; disjoint episodes, batch4/GPU, one combined score."""\nimport argparse,json,os,shutil,subprocess,sys,time\nfrom collections import Counter\nfrom pathlib import Path\nfrom umilife.core import read_jsonl,write_json,load_catalog,import_toolenv\nfrom umilife.evaluate import evaluate\n\ndef merge(root,work,ids):\n    predictions=[];traces=[];reports=[]\n    for rank in range(2):\n        part=work/f\'gpu{rank}\'\n        predictions.extend(read_jsonl(part/\'submission.jsonl\'));traces.extend(read_jsonl(part/\'traces.jsonl\'))\n        reports.append(json.loads((part/\'report.json\').read_text()))\n    assert len(predictions)==len(ids)==len(set(x[\'id\'] for x in predictions))\n    assert set(x[\'id\'] for x in predictions)==set(ids)\n    assert len(traces)==len(ids)==len(set(x[\'id\'] for x in traces))\n    lookup={x[\'id\']:x for x in predictions};trace_lookup={x[\'id\']:x for x in traces}\n    predictions=[lookup[i] for i in ids];traces=[trace_lookup[i] for i in ids]\n    for name,rows in [(\'submission\',predictions),(\'traces\',traces)]:\n        target=work/f\'{name}.jsonl\';temp=target.with_suffix(\'.tmp\')\n        temp.write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in rows));temp.replace(target)\n    catalog=load_catalog(root,\'train\');ToolEnv=import_toolenv(root,\'train\');references=[]\n    for row in read_jsonl(root/\'train/train.jsonl\'):\n        if row[\'id\'] not in lookup:continue\n        env=ToolEnv(\'train\',row[\'id\'],root=root)\n        for action in row[\'label\'][\'tool_calls\']:env.call(action[\'name\'],action[\'arguments\'])\n        references.append({\'id\':row[\'id\'],\'tool_calls\':row[\'label\'][\'tool_calls\'],\'state_hash\':env.get_state_hash()})\n    report={\'samples\':len(ids),\'metrics\':evaluate(predictions,references,catalog,traces),\n            \'asr\':json.loads((work/\'asr_report.json\').read_text()),\'workers\':reports,\n            \'termination_reasons\':dict(Counter(t[\'reason\'] for t in traces)),\n            \'runtime\':{\'independent_gpus\':2,\'batch_per_gpu\':4,\'total_concurrent_workflows\':8,\n                       \'cuda_graphs\':all(r[\'runtime\'][\'cuda_graphs\'] for r in reports),\'prefix_caching_per_worker\':[r[\'runtime\'][\'prefix_caching\'] for r in reports],\'max_revisions_per_role_per_decision\':1}}\n    write_json(work/\'report.json\',report);return report\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'data-root\',\'work\',\'model\',\'revision\',\'adapters\',\'ids-file\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--deadline-monotonic\',type=float);p.add_argument(\'--mode\',default=\'multi_local\');p.add_argument(\'--split\',default=\'validation\')\n    a=p.parse_args();assert a.mode==\'multi_local\' and a.split==\'validation\'\n    import torch\n    assert torch.cuda.device_count()==2,\'Select GPU T4 x2; no silent single-GPU fallback\'\n    work=Path(a.work);root=Path(a.data_root);ids=Path(a.ids_file).read_text().splitlines()\n    assert len(ids)==200 and len(set(ids))==200\n    processes=[];logs=[];counts=[None,None];ready=set()\n    try:\n        for rank in range(2):\n            part=work/f\'gpu{rank}\';part.mkdir(exist_ok=True)\n            (part/\'split.txt\').write_text(\'\\n\'.join(ids[rank::2])+\'\\n\')\n            for name in (\'asr_transcripts.json\',\'asr_report.json\'):shutil.copyfile(work/name,part/name)\n            env=os.environ.copy();env.update(CUDA_VISIBLE_DEVICES=str(rank),LIFEMATE_WORKER_GPU=str(rank),\n                TRITON_CACHE_DIR=f\'/tmp/lifemate_eval_triton_{rank}\',VLLM_CACHE_ROOT=f\'/tmp/lifemate_eval_vllm_{rank}\',OMP_NUM_THREADS=\'2\')\n            command=[sys.executable,\'-u\',\'-m\',\'umilife.infer_2gpu\',\'--data-root\',str(root),\'--work\',str(part),\n                \'--model\',a.model,\'--revision\',a.revision,\'--adapters\',a.adapters,\n                \'--ids-file\',str(part/\'split.txt\'),\'--split\',\'validation\',\'--mode\',\'multi_local\']\n            if a.deadline_monotonic:command+=[\'--deadline-monotonic\',str(a.deadline_monotonic)]\n            log=(part/\'worker.log\').open(\'a\',buffering=1);logs.append(log)\n            processes.append(subprocess.Popen(command,env=env,stdout=log,stderr=subprocess.STDOUT))\n            print(json.dumps({\'event\':\'worker_start\',\'gpu\':rank,\'samples\':len(ids[rank::2]),\'cuda_graphs\':True,\'prefix_caching_policy\':\'sm80_plus\'}),flush=True)\n        while any(p.poll() is None for p in processes):\n            for rank,process in enumerate(processes):\n                if process.poll() not in (None,0,75):\n                    print((work/f\'gpu{rank}\'/\'worker.log\').read_text()[-10000:],flush=True)\n                    raise RuntimeError(f\'GPU worker {rank} failed: {process.returncode}; inspect worker.log\')\n                if rank not in ready and (work/f\'gpu{rank}\'/\'preflight.json\').exists():\n                    ready.add(rank);settings=json.loads((work/f\'gpu{rank}\'/\'preflight.json\').read_text())\n                    print(json.dumps({\'event\':\'worker_ready\',\'gpu\':rank,\'runtime\':settings[\'runtime\']}),flush=True)\n                progress=work/f\'gpu{rank}\'/\'progress.json\'\n                if progress.exists():\n                    x=json.loads(progress.read_text());done=x[\'completed\']\n                    if done!=counts[rank]:\n                        counts[rank]=done;print(json.dumps({\'event\':\'progress\',\'gpu\':rank,**x}),flush=True)\n            time.sleep(2)\n        if any(p.returncode==75 for p in processes):raise SystemExit(75)\n        for rank,p in enumerate(processes):\n            if p.returncode!=0:\n                print((work/f\'gpu{rank}\'/\'worker.log\').read_text()[-14000:],flush=True)\n                raise RuntimeError(f\'GPU worker {rank} failed: {p.returncode}; inspect worker.log\')\n        report=merge(root,work,ids)\n        print(json.dumps({k:v for k,v in report[\'metrics\'].items() if k!=\'per_sample\'},indent=2),flush=True)\n    finally:\n        for p in processes:\n            if p.poll() is None:\n                p.terminate()\n                try:p.wait(timeout=10)\n                except subprocess.TimeoutExpired:p.kill();p.wait()\n        for log in logs:log.close()\nif __name__==\'__main__\':main()\n', 'umilife/prompt_policy.py': '"""Generic tool policy. No benchmark examples, held-out labels or per-sample rules."""\nfrom .typescript_schema import typescript_catalog\nfrom copy import deepcopy\nfrom datetime import datetime,timedelta\nimport hashlib,json\n\nPOLICY_VERSION=\'planner-caller-concise-ts-v2\'\nCOMMON="""You are LifeMate, a personal assistant using installed tools.\nAct only on the current request. History resolves references, not proof of execution in this turn. Preserve quantities, negation, conditions and exact text to send or save. Use the supplied clock and timezone. Never invent identifiers or tool results.\nOnly current observations confirm execution. Do not repeat successful effects, even with reworded arguments. Retry only with evidence of failure and a justified correction. Reuse unchanged read results; change strategy after an unhelpful lookup.\nTool contracts are TypeScript declarations: ? marks optional fields, literal unions specify enums, and comments describe constraints and identifier sources. These are interface documents, not executable code. Return one JSON object without markdown.\n"""\nPROMPTS={\n \'planner\':COMMON+"""You are Planner. Decide the next single step toward pending user objectives.\nChoose the installed tool, order dependent operations, evaluate conditions from available evidence, and stop when no further authorized operation is needed. If a condition or required identifier needs discovery, select an appropriate read tool first. Choose sys.ask_missing_info when required information cannot be resolved from context or supported lookups. An unresolved ID alone is not an unsupported request.\nWrite thought in concise English: normally one or two sentences, targeting at most 60 words. State the objective, essential selection constraints, and where Caller should obtain values. Refer to the request/history or a specific previous tool result. Caller resolves records, identifiers, field values and date formatting. Do not serialize the complete arguments or narrate all previous steps. Retain names, times or other user-supplied selectors when needed to distinguish pending objectives. Explain the decision rather than merely restating a tool name.\nFor an operation return {thought,next:"caller",tool}. For a completed/no-op turn or after requesting clarification return {thought,next:"summarizer"}; this terminal route does not require a trained Summarizer. For an unsupported request return {thought,next:"give_up"}. Never stop while a supported, authorized objective remains, repeat a confirmed write for a reply, or claim unconfirmed success.\n""",\n \'caller\':COMMON+"""You are Caller. Convert Planner\'s selected step into one valid API call.\nReturn exactly {name,arguments}; name must equal plan.tool. Planner owns routing and tool selection. Resolve values using the current request, history, context and previous tool observations. Treat plan.thought as guidance, not proof. Match the intended entity before taking its ID; use identifiers actually observed or explicitly supplied. Respect user qualifiers rather than choosing the first result. Normalize dates with the supplied clock/timezone and preserve exact text to send or save.\nRead the selected TypeScript contract, comments, required/optional fields, enums and alternative argument groups. Supply only supported values; omit optional fields without evidence. For sys.ask_missing_info, use the intended tool_name and exact missing parameter names from its contract. Never invent a required identifier or change the selected tool. Correct validation feedback without repeating confirmed effects. Output no thought, next or answer. The runtime executes your JSON.\n""",\n \'summarizer\':COMMON+"Return only {answer}, briefly grounded in actual observations. Do not claim unconfirmed success.",\n \'global\':COMMON+"Handle one ReAct step: return {thought,next,tool,action} for caller, {thought,next,answer} for summarizer, or {thought,next} for give_up. Keep the English thought concise; action contains {name,arguments}."\n}\nCOMPLETION_REVIEW="""Review this proposed ending against every objective in the current request. Has something only been looked up but not executed? Are you confusing history with current execution? Are there unfinished objectives, actual operation failures, or unresolved empty searches? Do not mistake business-state fields for execution failures. Do not propose repeating any confirmed effect just to verify it or generate a reply; summarize when all objectives are satisfied. A different argument spelling does not justify repeating a completed objective. Continue with caller when work remains. Prefer lookup before clarification. Summarize only with sufficient evidence or while waiting for clarification; give_up only beyond installed capabilities."""\n\ndef canonical(value):return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(\',\',\':\'))\ndef digest(value):return hashlib.sha256(canonical(value).encode()).hexdigest()\ndef obj(properties,required):return {\'type\':\'object\',\'properties\':properties,\'required\':required,\'additionalProperties\':False}\n\ndef parameter_schema(schema):\n    """Convert documented x-one-of to ordinary JSON Schema; preserve typed nesting."""\n    allowed={\'type\',\'properties\',\'required\',\'items\',\'enum\',\'format\',\'minimum\',\'maximum\',\n             \'exclusiveMinimum\',\'exclusiveMaximum\',\'minLength\',\'maxLength\',\'minItems\',\'maxItems\',\n             \'anyOf\',\'oneOf\',\'allOf\',\'additionalProperties\',\'description\'}\n    out={}\n    for key,value in schema.items():\n        if key not in allowed:continue\n        if key==\'properties\':out[key]={name:parameter_schema(prop) for name,prop in value.items()}\n        elif key==\'items\' and isinstance(value,dict):out[key]=parameter_schema(value)\n        elif key in (\'anyOf\',\'oneOf\',\'allOf\'):out[key]=[parameter_schema(x) for x in value]\n        else:out[key]=deepcopy(value)\n    if out.get(\'type\')==\'object\' and \'properties\' in out:out[\'additionalProperties\']=False\n    if schema.get(\'x-one-of\'):\n        # Full object alternatives are supported by both JSON Schema and LMFE.\n        base=out\n        out={\'anyOf\':[{**deepcopy(base),\'required\':list(dict.fromkeys(base.get(\'required\',[])+list(group)))}\n                      for group in schema[\'x-one-of\']]}\n    return out\n\ndef action_schema(specs,name):\n    arguments=parameter_schema(specs[name][\'parameters\'])\n    if name==\'sys.ask_missing_info\':\n        # Only public installed schemas provide these choices, never an answer label.\n        choices=[n for n,s in specs.items() if n!=name and s[\'parameters\'].get(\'properties\')]\n        if choices:arguments[\'properties\'][\'tool_name\'][\'enum\']=sorted(choices)\n        arguments[\'properties\'][\'params\'][\'minItems\']=1\n        fields=sorted({key for target in choices for key in specs[target][\'parameters\'][\'properties\']})\n        if fields:arguments[\'properties\'][\'params\'][\'items\'][\'enum\']=fields\n    return obj({\'name\':{\'type\':\'string\',\'enum\':[name]},\'arguments\':arguments},[\'name\',\'arguments\'])\n\ndef route_schema(specs,global_mode=False):\n    thought={\'type\':\'string\',\'maxLength\':1024};branches=[]\n    if specs:\n        if global_mode:\n            for name in sorted(specs):\n                branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                    \'tool\':{\'type\':\'string\',\'enum\':[name]},\'action\':action_schema(specs,name)},[\'thought\',\'next\',\'tool\',\'action\']))\n        else:\n            branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                \'tool\':{\'type\':\'string\',\'enum\':sorted(specs)}},[\'thought\',\'next\',\'tool\']))\n    done={\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'summarizer\']}}\n    if global_mode:done[\'answer\']={\'type\':\'string\',\'maxLength\':1024}\n    branches.append(obj(done,list(done)))\n    branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'give_up\']}},[\'thought\',\'next\']))\n    return {\'anyOf\':branches}\n\ndef compact_catalog(specs):\n    fields=(\'type\',\'enum\',\'x-ref\',\'format\',\'items\',\'minimum\',\'maximum\')\n    return [{\'name\':name,\'kind\':spec.get(\'kind\'),\'description\':spec[\'description\'],\n             \'parameters\':{key:{k:deepcopy(v) for k,v in prop.items() if k in fields}\n                           for key,prop in spec[\'parameters\'].get(\'properties\',{}).items()},\n             \'required\':spec[\'parameters\'].get(\'required\',[]),\n             **({\'one_of\':spec[\'parameters\'][\'x-one-of\']} if spec[\'parameters\'].get(\'x-one-of\') else {})}\n            for name,spec in sorted(specs.items())]\n\ndef reference_clock(context):\n    result={}\n    try:\n        now=datetime.fromisoformat(context[\'current_time\'])\n        result={\'today\':now.date().isoformat(),\'tomorrow\':(now+timedelta(days=1)).date().isoformat(),\n                \'day_after_tomorrow\':(now+timedelta(days=2)).date().isoformat(),\n                \'weekday_iso\':now.isoweekday(),\'timezone\':context.get(\'timezone\')}\n    except (KeyError,TypeError,ValueError):pass\n    return result\n\ndef build_messages(role,row,text,specs,trajectory,plan=None,feedback=None):\n    # Explicit provenance allowlist. Current labels/final state/reference transcript never enter here.\n    context={k:deepcopy(row[\'context\'][k]) for k in (\'current_time\',\'timezone\',\'location\',\'user_profile\') if k in row[\'context\']}\n    history=[{k:deepcopy(turn[k]) for k in (\'role\',\'text\',\'tool_calls\',\'tool_results\') if k in turn}\n             for turn in row.get(\'history\',[])]\n    content={\'context\':context,\'reference_clock\':reference_clock(context),\'history\':history,\n             \'request\':text,\'trajectory\':deepcopy(trajectory)}\n    if role in (\'planner\',\'global\'):content[\'tools_ts\']=typescript_catalog(specs)\n    elif role==\'caller\':\n        selected=(plan or {}).get(\'tool\');assert selected in specs,\'Planner selected an unavailable tool\'\n        content.update(plan=deepcopy(plan),selected_tool_ts=typescript_catalog({selected:specs[selected]},detailed=True))\n        if selected==\'sys.ask_missing_info\':\n            content[\'missing_information_tools_ts\']=typescript_catalog({name:spec for name,spec in specs.items()\n                if name!=selected and spec[\'parameters\'].get(\'properties\')})\n    else:content[\'plan\']=deepcopy(plan)\n    if feedback is not None:content[\'validation_feedback\']=deepcopy(feedback)\n    # Keep the original request at the end, after schemas/observations/feedback.\n    content.pop(\'request\');content[\'request\']=text\n    return [{\'role\':\'system\',\'content\':PROMPTS[role]},\n            {\'role\':\'user\',\'content\':json.dumps(content,ensure_ascii=False,separators=(\',\',\':\'))}]\n\ndef request_schema(role,specs,plan=None):\n    if role==\'caller\':return action_schema(specs,plan[\'tool\'])\n    if role in (\'planner\',\'global\'):return route_schema(specs,role==\'global\')\n    return obj({\'answer\':{\'type\':\'string\',\'maxLength\':1024}},[\'answer\'])\n\ndef validate_meta_action(action,specs):\n    if action[\'name\']!=\'sys.ask_missing_info\':return None\n    args=action[\'arguments\'];target=args.get(\'tool_name\');params=args.get(\'params\',[])\n    if target not in specs or target==\'sys.ask_missing_info\':return \'Clarification target must be another installed tool\'\n    # An optional API parameter can still be necessary to express the user\'s intent.\n    fields=set(specs[target][\'parameters\'].get(\'properties\',{}))\n    if not params or len(params)!=len(set(params)) or not set(params)<=fields:\n        return \'params must contain unique parameter names from the target tool contract\'\n    return None\n\ndef repeated_read(action,specs,trajectory):\n    if specs[action[\'name\']].get(\'kind\')!=\'read\':return False\n    for old in reversed(trajectory):\n        if specs.get(old[\'action\'][\'name\'],{}).get(\'kind\')==\'write\':return False\n        if canonical(old[\'action\'])==canonical(action):return True\n    return False\n\nPOLICY_AUDIT={\'policy_version\':POLICY_VERSION,\'static_prompts_sha256\':digest(PROMPTS),\n    \'completion_review_sha256\':digest(COMPLETION_REVIEW),\'few_shot_examples\':0,\n    \'sample_specific_rules\':False,\'gold_inputs_to_model\':False,\n    \'allowed_input_sources\':[\'current ASR request\',\'ASR user history\',\'provided context\',\n        \'installed public tool schemas\',\'observations from executed tools\',\'model-generated plan/feedback\']}\n', 'umilife/runtime_options.py': '"""Pinned vLLM 0.9.2 / Triton3.3 compatibility policy, independent of Torch."""\ndef engine_options(capability):\n    # SM75 prefix-prefill fails compilation on the observed FP16 path. Keep graphs.\n    prefix=tuple(capability)>=(8,0)\n    return {\'cuda_graphs\':True,\'prefix_caching\':prefix,\n            \'prefix_caching_policy\':\'sm80_plus\',\n            \'prefix_caching_reason\':(\'enabled_on_sm80_plus\' if prefix else\n                \'disabled_on_pre_sm80_due_to_prefix_prefill_triton_conversion_failure\'),\n            \'compute_capability\':list(capability)}\n', 'umilife/transcribe_supplied.py': '"""Recognize current/history audio using only the supplied Zipformer2 ONNX files."""\nimport argparse,hashlib,json,math,time\nfrom pathlib import Path\nimport numpy as np\nimport soundfile as sf\nfrom scipy.signal import resample_poly\nimport sherpa_onnx\n\ndef digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef transcribe(root,work,model,limit=None,threads=4,split="train"):\n    root=Path(root);work=Path(work);model=Path(model)\n    manifest=json.loads((model/\'model_manifest.json\').read_text())\n    assert manifest[\'model_type\']==\'zipformer2\' and not manifest[\'streaming\']\n    for name,info in manifest[\'files\'].items():assert digest(model/name)==info[\'sha256\'],f\'Wrong ASR file: {name}\'\n    fingerprint=digest(model/\'model_manifest.json\')\n    valid=set((work/\'split.txt\').read_text().splitlines());assert len(valid)>0\n    rows=[json.loads(line) for line in (root/split/f\'{split}.jsonl\').open()]\n    selected=[r for r in rows if r[\'id\'] in valid];assert len(selected)==len(valid)\n    if limit:selected=selected[:limit]\n    audio=[]\n    for row in selected:\n        audio.append(row[\'audio\'])\n        audio.extend(t[\'audio\'] for t in row.get(\'history\',[]) if t[\'role\']==\'user\')\n    audio=list(dict.fromkeys(audio))\n    config={\'model_manifest_sha256\':fingerprint,\'split_sha256\':digest(work/\'split.txt\'),\n        \'split\':split,\'selected_ids\':[r[\'id\'] for r in selected],\'sample_rate\':16000,\'decoding_method\':\'greedy_search\',\n        \'provider\':\'cpu\',\'sherpa_onnx\':sherpa_onnx.__version__,\'source\':\'supplied_asr_model\'}\n    config_path=work/\'asr_config.json\'\n    if config_path.exists():assert json.loads(config_path.read_text())==config,\'Stale ASR cache configuration\'\n    else:config_path.write_text(json.dumps(config,indent=2))\n    cache_path=work/\'asr_transcripts.jsonl\';cache={}\n    if cache_path.exists():\n        for line in cache_path.read_text().splitlines():\n            record=json.loads(line);assert record[\'model_manifest_sha256\']==fingerprint\n            cache[record[\'audio\']]=record\n    recognizer=sherpa_onnx.OfflineRecognizer.from_transducer(\n        encoder=str(model/\'encoder.onnx\'),decoder=str(model/\'decoder.onnx\'),joiner=str(model/\'joiner.onnx\'),\n        tokens=str(model/\'tokens.txt\'),num_threads=threads,sample_rate=16000,feature_dim=80,\n        decoding_method=\'greedy_search\',provider=\'cpu\')\n    started=time.monotonic()\n    with cache_path.open(\'a\',encoding=\'utf-8\') as output:\n        for index,name in enumerate(audio,1):\n            path=(root/split/name).resolve();assert path.is_relative_to((root/split).resolve())\n            audio_sha=digest(path)\n            if name in cache:\n                assert cache[name][\'audio_sha256\']==audio_sha,\'Audio changed after transcription\'\n                continue\n            tick=time.monotonic()\n            try:\n                samples,sample_rate=sf.read(path,dtype=\'float32\',always_2d=True)\n                samples=samples.mean(axis=1)\n                assert len(samples)>0 and np.isfinite(samples).all(),\'Empty or non-finite waveform\'\n                duration=len(samples)/sample_rate\n                if sample_rate!=16000:\n                    factor=math.gcd(sample_rate,16000)\n                    samples=resample_poly(samples,16000//factor,sample_rate//factor)\n                stream=recognizer.create_stream()\n                stream.accept_waveform(16000,np.ascontiguousarray(samples,dtype=np.float32))\n                recognizer.decode_stream(stream)\n                record={\'audio\':name,\'text\':stream.result.text.strip(),\'audio_sha256\':audio_sha,\n                    \'model_manifest_sha256\':fingerprint,\'elapsed_s\':time.monotonic()-tick,\'duration_s\':duration,\n                    \'source\':\'supplied_asr_model\',\'provider\':\'cpu\'}\n            except Exception as exc:\n                (work/\'asr_error.json\').write_text(json.dumps({\'audio\':name,\'error\':str(exc)}))\n                raise\n            output.write(json.dumps(record,ensure_ascii=False)+\'\\n\');output.flush();cache[name]=record\n            print(json.dumps({\'stage\':\'asr\',\'completed\':index,\'total_audio\':len(audio),\n                \'validation_samples\':len(selected),\'audio\':name,\'elapsed_s\':record[\'elapsed_s\']},ensure_ascii=False),flush=True)\n    assert set(audio)<=cache.keys()\n    transcripts={name:cache[name][\'text\'] for name in audio}\n    (work/\'asr_transcripts.json\').write_text(json.dumps(transcripts,ensure_ascii=False,indent=2))\n    report={\'status\':\'complete\',\'validation_samples\':len(selected),\'unique_audio\':len(audio),\n        \'current_and_history_audio\':True,\'oracle_fallback\':False,\'training\':False,\n        \'empty_transcripts\':sum(not value.strip() for value in transcripts.values()),\n        \'wall_s_this_run\':time.monotonic()-started,\n        \'audio_duration_s\':sum(cache[name][\'duration_s\'] for name in audio),\n        \'initializer_elements\':sum(info.get(\'initializer_elements\',0) for info in manifest[\'files\'].values()),\n        \'model_manifest_sha256\':fingerprint,\'provider\':\'cpu\',\'model_type\':\'zipformer2\'}\n    (work/\'asr_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'ASR_COMPLETE\',json.dumps(report),flush=True)\n    return report\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    for name in (\'data-root\',\'work\',\'model-dir\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--split\',choices=[\'train\',\'public_test\'],default=\'train\')\n    p.add_argument(\'--limit\',type=int);p.add_argument(\'--threads\',type=int,default=4)\n    a=p.parse_args();transcribe(a.data_root,a.work,a.model_dir,a.limit,a.threads,a.split)\n', 'umilife/typescript_schema.py': '"""Compact TypeScript interfaces derived solely from the installed public catalog."""\nimport json\n\ndef literal(v):return json.dumps(v,ensure_ascii=False,separators=(\',\',\':\'))\ndef ts_type(p):\n    if \'enum\' in p:return \' | \'.join(literal(v) for v in p[\'enum\'])\n    t=p.get(\'type\')\n    if isinstance(t,list):return \' | \'.join(ts_type({**p,\'type\':v}) for v in t)\n    if t==\'array\':return \'(\'+ts_type(p.get(\'items\',{}))+\')[]\'\n    if t==\'object\':\n        fields=p.get(\'properties\')\n        if fields is None:return \'Record<string, unknown>\'\n        base=\'{ \'+ \'; \'.join(literal(k)+(\'\' if k in p.get(\'required\',[]) else \'?\')+\': \'+ts_type(v) for k,v in fields.items())+\' }\'\n        if p.get(\'x-one-of\'):\n            alt=[\'{ \'+\'; \'.join(literal(k)+\': \'+ts_type(fields[k]) for k in group)+\' }\' for group in p[\'x-one-of\']]\n            return base+\' & (\'+\' | \'.join(alt)+\')\'\n        return base\n    return {\'string\':\'string\',\'integer\':\'number\',\'number\':\'number\',\'boolean\':\'boolean\',\'null\':\'null\'}.get(t,\'unknown\')\n\ndef metadata(p,path=\'\',detailed=False):\n    result=[]\n    for k,v in p.items():\n        if k in (\'properties\',\'items\',\'required\',\'type\',\'enum\',\'x-one-of\'):continue\n        if k==\'description\' and not detailed:continue\n        result.append((path+\'.\' if path else \'\')+k+\'=\'+literal(v))\n    if p.get(\'type\')==\'integer\':result.append(path+\':integer\')\n    for k,v in p.get(\'properties\',{}).items():result.extend(metadata(v,(path+\'.\' if path else \'\')+k,detailed))\n    if p.get(\'items\'):result.extend(metadata(p[\'items\'],path+\'[]\',detailed))\n    return result\n\ndef typescript_catalog(specs,detailed=False):\n    lines=[\'interface InstalledTools {\']\n    for name,s in sorted(specs.items()):\n        comment=\'; \'.join([s.get(\'kind\',\'\'),s.get(\'description\',\'\'),*metadata(s[\'parameters\'],detailed=detailed)])\n        if s.get(\'returns\') and detailed:comment+=\'; returns=\'+str(s[\'returns\'])\n        comment=comment.replace(\'*/\',\'* /\').replace(\'\\n\',\' \')\n        lines.append(\'  /** \'+comment+\' */\')\n        lines.append(\'  \'+literal(name)+\'(args: \'+ts_type(s[\'parameters\'])+\'): unknown;\')\n    return \'\\n\'.join(lines+[\'}\'])\n'}
SOURCE_SHA='08d8ed8080382e13bed32e0429506d3405cf234721ed1ee896912d60d0e8adef'
assert hashlib.sha256(json.dumps(PAYLOAD,sort_keys=True).encode()).hexdigest()==SOURCE_SHA
for name,content in PAYLOAD.items():
    path=SOURCE/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(content)
assert hashlib.sha256((SOURCE/'split_200.txt').read_bytes()).hexdigest()=='e810b418d7018437a16892350e01cbee9526cf43ab4a190101433d4508498d2f'


In [ ]:
sys.path.insert(0,str(SOURCE))
from umilife.paths import find_data_root
DATA_ROOT,archive=find_data_root('/kaggle/input')
if DATA_ROOT is None:
    destination=Path('/tmp/v6_competition_data');destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        assert all((destination/m.filename).resolve().is_relative_to(destination.resolve()) for m in z.infolist())
        z.extractall(destination)
    DATA_ROOT,_=find_data_root(destination)
assert hashlib.sha256((DATA_ROOT/'train/train.jsonl').read_bytes()).hexdigest()=='24c42c79fbe8db932fe9817d041c0d0ca422ac8177b413cc1be0cda1fd8941a8'
def input_dir(slug,marker):
    candidates=[p.parent for p in Path('/kaggle/input').rglob(marker) if p.parent.name==slug]
    return next((p for p in candidates if (p/marker).is_file()),None)
ADAPTERS=input_dir('lifemate-concise-planner-caller-adapters','checkpoint_manifest.json')
ASR_MODEL=input_dir('lifemate-concise-validation-asr','model_manifest.json')
assert ADAPTERS is not None and ASR_MODEL is not None,'Attach completed v6 adapters and supplied ASR datasets'
manifest=json.loads((ADAPTERS/'checkpoint_manifest.json').read_text())
assert manifest['roles']==['planner','caller']
missing=[name for name in manifest['files'] if not (ADAPTERS/name).is_file()]
assert not missing, f'Incomplete checkpoint input: {missing}. Update lifemate-concise-planner-caller-adapters to dataset version 2 or later.'
print('Checkpoint input:',ADAPTERS,'Files:',len(manifest['files']))
for name,expected in manifest['files'].items():assert hashlib.sha256((ADAPTERS/name).read_bytes()).hexdigest()==expected,name
from umilife.prompt_policy import PROMPTS,digest,POLICY_VERSION
for role in ('planner','caller'):
    meta=json.loads((ADAPTERS/role/'lineage.json').read_text())
    assert meta['prompt_sha256']==digest(PROMPTS) and meta['model']==MODEL and meta['revision']==REVISION
config={'mode':MODE,'model':MODEL,'revision':REVISION,'source_sha':SOURCE_SHA,
        'checkpoint_manifest_sha':hashlib.sha256((ADAPTERS/'checkpoint_manifest.json').read_bytes()).hexdigest(),
        'prompt_sha':digest(PROMPTS),'policy':POLICY_VERSION,'batch_size_per_gpu':4,'gpu_workers':2,'guided_decoding':False,'max_decisions':7,'max_tool_calls':7,'cuda_graphs':True,'prefix_caching_policy':'sm80_plus','revisions_per_role':1,'asr':True,'split_sha256':hashlib.sha256((SOURCE/'split_200.txt').read_bytes()).hexdigest()}
if (WORK/'config.json').exists():assert json.loads((WORK/'config.json').read_text())==config,'Different run settings; use a fresh RUN_NAME'
else:(WORK/'config.json').write_text(json.dumps(config,indent=2))
print('Checkpoint roles:',manifest['roles'],'Mode:',MODE)
ids=(SOURCE/'split_200.txt').read_text().splitlines()
assert len(ids)==len(set(ids))==200
train_ids=set((SOURCE/'adapter_train_ids.txt').read_text().splitlines());valid_ids=set((SOURCE/'adapter_valid_ids.txt').read_text().splitlines())
audit={'samples':len(ids),'adapter_train_overlap':sorted(set(ids)&train_ids),'adapter_valid_overlap':sorted(set(ids)&valid_ids),'note':'Requested split includes adapter training episodes; aggregate score is not held-out validation.'}
assert len(audit['adapter_train_overlap'])==168 and len(audit['adapter_valid_overlap'])==32
(WORK/'split_audit.json').write_text(json.dumps(audit,indent=2))
print('SPLIT AUDIT:',len(audit['adapter_train_overlap']),'train overlap;',len(audit['adapter_valid_overlap']),'old valid overlap')


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','uv==0.9.10'],check=True)
UV=[sys.executable,'-m','uv'];os.environ['UV_NATIVE_TLS']='true'
def runtime(name,lock,gpus):
    path=Path('/tmp')/name;python=str(path/'bin/python')
    if not Path(python).exists():subprocess.run([*UV,'venv','--python','3.12','--python-preference','only-managed','--seed',str(path)],check=True)
    subprocess.run([*UV,'pip','sync','--index-strategy','unsafe-best-match','--python',python,str(SOURCE/lock)],check=True)
    subprocess.run([*UV,'pip','check','--python',python],check=True)
    purelib=Path(subprocess.check_output([python,'-I','-c',"import sysconfig;print(sysconfig.get_path('purelib'))"],text=True).strip())
    env=os.environ.copy()
    for key in ('PYTHONPATH','PYTHONHOME'):env.pop(key,None)
    env.update(CUDA_VISIBLE_DEVICES=gpus,PYTHONNOUSERSITE='1',TOKENIZERS_PARALLELISM='false',
        VLLM_USE_V1='0',VLLM_ATTENTION_BACKEND='XFORMERS',VLLM_WORKER_MULTIPROC_METHOD='spawn',
        VLLM_USAGE_STATS_ENABLED='0',DO_NOT_TRACK='1',HF_HOME='/tmp/lifemate_hf_cache')
    env['LD_LIBRARY_PATH']=':'.join([str(p) for p in sorted((purelib/'nvidia').glob('*/lib'))]+[os.environ.get('LD_LIBRARY_PATH','')])
    return python,env
INFER_PYTHON,INFER_ENV=runtime('lifemate_vllm092_env','requirements_infer_2gpu.lock','0,1')
ASR_PYTHON,ASR_ENV=runtime('lifemate_supplied_asr_env','requirements_asr_2gpu.lock','')
subprocess.run([INFER_PYTHON,'-c',"import torch,vllm;assert torch.cuda.device_count()==2;assert vllm.__version__=='0.9.2'"],env=INFER_ENV,check=True)


In [ ]:
def run_split(name):
    output=WORK/name;output.mkdir(exist_ok=True)
    if name=='validation':ids=(SOURCE/'split_200.txt').read_text().splitlines();assert len(ids)==200;split='train'
    else:raise ValueError('Validation only')
    assert len(ids)==len(set(ids))
    (output/'split.txt').write_text('\n'.join(ids)+'\n')
    subprocess.run([ASR_PYTHON,'-u','-m','umilife.transcribe_supplied','--data-root',str(DATA_ROOT),
        '--work',str(output),'--model-dir',str(ASR_MODEL),'--split',split],cwd=SOURCE,env=ASR_ENV,check=True)
    subprocess.run([INFER_PYTHON,'-u','-m','umilife.parallel_validation','--data-root',str(DATA_ROOT),
        '--work',str(output),'--model',MODEL,'--revision',REVISION,'--adapters',str(ADAPTERS),
        '--ids-file',str(output/'split.txt'),'--split',name,'--mode',MODE,
        '--deadline-monotonic',str(SESSION_STARTED+SESSION_HOURS*3600)],cwd=SOURCE,env=INFER_ENV,check=True)
    report=json.loads((output/'report.json').read_text());assert report['samples']==len(ids)
    report['split_audit']=audit
    (output/'report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
    return report
validation=run_split('validation')
assert validation['samples']==200
print('VALIDATION:',json.dumps(validation['metrics'],ensure_ascii=False,indent=2))
